# Tarot Interaction Restyler

Edit writing styles for your existing `tarot_data_full.json` **without regenerating everything**.

Three approaches (all free):
1. **Hugging Face Inference API** — free tier, open models (Qwen, Mistral, etc.)
2. **Gemini free tier** — if you still have RPD budget
3. **Manual style rules** — regex/Python transforms (no API needed)

### How it works
- Load your full dataset from Google Drive
- Select which cards/pairs/dynamics to restyle
- Apply a new style prompt or transformation
- Save back to Drive (original is backed up automatically)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import shutil
import time
import re

DRIVE_FOLDER = "/content/drive/MyDrive/tarot-generator"
INPUT_FILE = os.path.join(DRIVE_FOLDER, "tarot_data_full.json")

# Load the full dataset
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

total_interactions = sum(len(c['interactions']) for c in data)
print(f"Loaded {len(data)} cards, {total_interactions} interactions")
print(f"Source: {INPUT_FILE}")

## 1. Explore & Select What to Restyle

Browse cards and interactions before deciding what to change.

In [ ]:
# ============================================================
# HELPERS: browse and select
# ============================================================

card_by_id = {c['id']: c for c in data}
card_by_name = {c['name']: c for c in data}

def list_cards():
    """Show all cards with IDs."""
    for c in data:
        print(f"  {c['id']:>3s}  {c['name']}")

def show_interaction(source_id, target_id):
    """Show the 5 dynamics for a specific pair."""
    card = card_by_id[source_id]
    for inter in card['interactions']:
        if inter['target_id'] == target_id:
            print(f"\n{card['name']} -> {inter['target_name']}\n")
            for k, v in inter['dynamics'].items():
                words = len(v.split())
                print(f"  [{k}] ({words} words)")
                print(f"    {v}\n")
            return
    print(f"Pair {source_id}->{target_id} not found")

def show_card_sample(card_id, n=3):
    """Show first n interactions for a card."""
    card = card_by_id[card_id]
    print(f"\n=== {card['name']} (id={card['id']}) ===")
    print(f"Total interactions: {len(card['interactions'])}\n")
    for inter in card['interactions'][:n]:
        print(f"  -> {inter['target_name']}")
        eng = inter['dynamics']['engendrar']
        print(f"     engendrar: {eng[:150]}...\n")

# Show a sample
show_card_sample("1", n=3)
print("\n--- Full interaction example ---")
show_interaction("1", "2")

In [ ]:
# ============================================================
# SELECTION: choose what to restyle
# ============================================================

# Option A: Specific pairs (source_id, target_id)
SELECTED_PAIRS = [
    ("1", "2"),   # O Mago -> A Papisa
    ("1", "3"),   # O Mago -> A Imperatriz
    # Add more pairs here...
]

# Option B: All interactions for specific cards
SELECTED_CARDS = [
    # "1",   # All of O Mago's interactions
    # "13",  # All of A Morte's interactions
]

# Option C: Specific dynamics only
SELECTED_DYNAMICS = [
    # "engendrar",
    # "conflito",
    # "estagnar",
    # "regressar",
    # "necessitar",
]

def collect_targets():
    """Build list of (card, interaction, dynamic_keys) to restyle."""
    targets = []
    dynamics_filter = SELECTED_DYNAMICS or ['engendrar','conflito','estagnar','regressar','necessitar']

    for card in data:
        for inter in card['interactions']:
            pair = (card['id'], inter['target_id'])
            if SELECTED_PAIRS and pair not in SELECTED_PAIRS:
                if SELECTED_CARDS and card['id'] not in SELECTED_CARDS:
                    continue
                elif not SELECTED_CARDS:
                    continue
            targets.append({
                'card': card,
                'interaction': inter,
                'dynamics': dynamics_filter,
            })
    return targets

targets = collect_targets()
print(f"Selected {len(targets)} interactions to restyle")
if targets:
    t = targets[0]
    print(f"  First: {t['card']['name']} -> {t['interaction']['target_name']}")
    print(f"  Dynamics: {t['dynamics']}")

## 2. Choose Your Restyling Engine

Pick **one** of the three options below.

### Option A: Hugging Face Inference API (FREE)

Uses open models like Qwen 2.5 72B or Mistral via HF's free serverless API.

1. Get a free token at https://huggingface.co/settings/tokens
2. Add it as Colab secret `HF_TOKEN`

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import InferenceClient
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = ""  # <-- Paste your token here if not using Colab Secrets

assert HF_TOKEN, "Get a free token at https://huggingface.co/settings/tokens"

# Free serverless models (no billing needed):
#   - "Qwen/Qwen2.5-72B-Instruct"      : excellent for Portuguese, large context
#   - "mistralai/Mistral-Small-24B-Instruct-2501" : fast, good quality
#   - "meta-llama/Llama-3.3-70B-Instruct": strong general purpose
HF_MODEL = "Qwen/Qwen2.5-72B-Instruct"

hf_client = InferenceClient(token=HF_TOKEN)

# Quick test
test = hf_client.chat_completion(
    model=HF_MODEL,
    messages=[{"role": "user", "content": "Diga 'ola' em portugues poetico, em 1 frase."}],
    max_tokens=100,
)
print(f"Model: {HF_MODEL}")
print(f"Test: {test.choices[0].message.content}")
print("HF Inference API ready!")

### Option B: Gemini Free Tier

If you still have daily RPD budget, you can use Gemini for restyling too.

In [ ]:
# Uncomment to use Gemini instead of HF:

# !pip install -q google-genai
# from google import genai
# from google.genai import types
# from google.colab import userdata
#
# API_KEY = userdata.get('GEMINI_API_KEY')
# gemini_client = genai.Client(api_key=API_KEY)
# GEMINI_MODEL = "gemini-2.5-flash-lite"
# USE_GEMINI = True
# print("Gemini ready!")

## 3. Define Your New Style

Edit the style prompt below. This controls how the text gets rewritten.

In [ ]:
# ============================================================
# STYLE PRESETS - pick one or write your own
# ============================================================

STYLE_PRESETS = {
    "poetico_intenso": {
        "name": "Poetico Intenso",
        "instruction": """Reescreva o texto em portugues brasileiro com um estilo ALTAMENTE POETICO 
e LIRICO. Use metaforas ricas, linguagem evocativa e ritmo cadenciado. 
Mantenha o significado simbolico do taro mas eleve a linguagem ao nivel de poesia.
Use 2-3 frases (50-80 palavras)."""
    },
    "matter_of_fact": {
        "name": "Matter of Fact",
        "instruction": """Reescreva o texto em portugues brasileiro com tom MATTER OF FACT: direto,
descritivo e sem ornamentos. Descreva o que acontece na dinamica como se fosse
um relato objetivo de causa e efeito psicologico ou situacional. Sem metaforas
florescentes, sem linguagem esoterica. Como um terapeuta ou consultor descreveria
a situacao. Use frases curtas e declarativas. 2-3 frases (50-80 palavras)."""
    },
    "direto_pratico": {
        "name": "Direto e Pratico",
        "instruction": """Reescreva o texto em portugues brasileiro CLARO e DIRETO.
Use linguagem acessivel, sem jargao esoterico excessivo. Foque no significado 
pratico e aplicavel da interacao entre as cartas. Seja conciso mas informativo.
Use 2-3 frases (50-80 palavras)."""
    },
    "mistico_sombrio": {
        "name": "Mistico e Sombrio",
        "instruction": """Reescreva o texto em portugues brasileiro com tom MISTICO e SOMBRIO.
Evoque o misterio, a profundidade dos arquetipos e as sombras do inconsciente.
Use linguagem densa e atmosferica, como um oraculo antigo.
Use 2-3 frases (50-80 palavras)."""
    },
    "contemporaneo": {
        "name": "Contemporaneo",
        "instruction": """Reescreva o texto em portugues brasileiro CONTEMPORANEO.
Traduza os simbolismos do taro para situacoes e linguagem do dia a dia moderno.
Mantenha a profundidade mas use referencias atuais e tom conversacional-elegante.
Use 2-3 frases (50-80 palavras)."""
    },
    "custom": {
        "name": "Custom",
        "instruction": """Reescreva o texto em portugues brasileiro com o seguinte estilo:
[DESCREVA SEU ESTILO AQUI]
Use 2-3 frases (50-80 palavras)."""
    },
}

# ============ CHOOSE YOUR STYLE ============
CHOSEN_STYLE = "matter_of_fact"  # Change this!
# ============================================

style = STYLE_PRESETS[CHOSEN_STYLE]
print(f"Style: {style['name']}")
print(f"Instruction: {style['instruction'][:200]}...")

In [ ]:
# ============================================================
# RESTYLING ENGINE
# ============================================================

USE_GEMINI = False  # Set True if using Gemini instead of HF

def restyle_text(original_text, card_name, target_name, dynamic_name, style_instruction):
    """Restyle a single dynamic text using the chosen engine."""
    prompt = f"""Voce e um especialista em Taro de Marselha.

{style_instruction}

Contexto: Dinamica "{dynamic_name}" entre {card_name} e {target_name}.

Texto original:
\"\"\"
{original_text}
\"\"\"

Reescreva APENAS o texto, sem explicacoes ou prefacios. Mantenha os nomes e numerais das cartas."""

    if USE_GEMINI:
        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.9, max_output_tokens=500),
        )
        return response.text.strip().strip('"')
    else:
        response = hf_client.chat_completion(
            model=HF_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
            temperature=0.9,
        )
        return response.choices[0].message.content.strip().strip('"')

# Quick test
if targets:
    t = targets[0]
    original = t['interaction']['dynamics'][t['dynamics'][0]]
    print(f"Original ({len(original.split())} words):")
    print(f"  {original[:200]}...\n")

    restyled = restyle_text(
        original,
        t['card']['name'],
        t['interaction']['target_name'],
        t['dynamics'][0],
        style['instruction']
    )
    print(f"Restyled ({len(restyled.split())} words):")
    print(f"  {restyled}")

## 4. Run the Restyling

Processes only the selected pairs/dynamics. Safe to re-run.

In [ ]:
# ============================================================
# BACKUP & RESTYLE
# ============================================================

# Auto-backup before any changes
backup_file = os.path.join(DRIVE_FOLDER, "tarot_data_full_backup.json")
if not os.path.exists(backup_file):
    shutil.copy2(INPUT_FILE, backup_file)
    print(f"Backup saved: {backup_file}")
else:
    print(f"Backup already exists: {backup_file}")

# Restyle checkpoint (resumes if interrupted)
RESTYLE_CHECKPOINT = os.path.join(DRIVE_FOLDER, "restyle_checkpoint.json")

def load_restyle_progress():
    if os.path.exists(RESTYLE_CHECKPOINT):
        with open(RESTYLE_CHECKPOINT, 'r') as f:
            return json.load(f)
    return {"completed": []}

def save_restyle_progress(progress):
    with open(RESTYLE_CHECKPOINT, 'w') as f:
        json.dump(progress, f)

progress = load_restyle_progress()
completed_keys = set(progress['completed'])

total = sum(len(t['dynamics']) for t in targets)
skipped = 0
done = 0
errors = 0

print(f"\nRestyling {total} dynamics across {len(targets)} interactions")
print(f"Style: {style['name']}")
print(f"{'='*50}\n")

for i, t in enumerate(targets):
    card = t['card']
    inter = t['interaction']
    card_name = card['name']
    target_name = inter['target_name']

    for dyn_key in t['dynamics']:
        key = f"{card['id']}_{inter['target_id']}_{dyn_key}"
        if key in completed_keys:
            skipped += 1
            continue

        try:
            original = inter['dynamics'][dyn_key]
            restyled = restyle_text(original, card_name, target_name, dyn_key, style['instruction'])

            # Update in-memory data
            inter['dynamics'][dyn_key] = restyled
            done += 1

            # Track progress
            progress['completed'].append(key)
            completed_keys.add(key)

            if done % 5 == 0:
                # Save progress every 5 completions
                save_restyle_progress(progress)
                with open(INPUT_FILE, 'w', encoding='utf-8') as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)

            print(f"[{done + skipped}/{total}] {card_name} -> {target_name} [{dyn_key}] OK")

            # Rate limiting (HF free tier is ~30 RPM, be gentle)
            time.sleep(2.5)

        except Exception as e:
            errors += 1
            print(f"  ERROR: {str(e)[:100]}")
            time.sleep(5)

# Final save
save_restyle_progress(progress)
with open(INPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"\n{'='*50}")
print(f"Done! Restyled: {done}, Skipped: {skipped}, Errors: {errors}")
print(f"Saved to: {INPUT_FILE}")

## 5. Compare Before & After

In [ ]:
# ============================================================
# SIDE-BY-SIDE COMPARISON
# ============================================================

if os.path.exists(backup_file):
    with open(backup_file, 'r', encoding='utf-8') as f:
        original_data = json.load(f)
    orig_by_id = {c['id']: c for c in original_data}

    # Compare first few restyled interactions
    count = 0
    for t in targets:
        if count >= 3:
            break
        card = t['card']
        inter = t['interaction']
        orig_card = orig_by_id.get(card['id'])
        if not orig_card:
            continue

        orig_inter = None
        for oi in orig_card['interactions']:
            if oi['target_id'] == inter['target_id']:
                orig_inter = oi
                break
        if not orig_inter:
            continue

        print(f"\n{'='*60}")
        print(f"{card['name']} -> {inter['target_name']}")
        print(f"{'='*60}")

        for dyn_key in t['dynamics'][:2]:  # Show first 2 dynamics
            print(f"\n  [{dyn_key}]")
            print(f"  BEFORE: {orig_inter['dynamics'][dyn_key][:200]}...")
            print(f"  AFTER:  {inter['dynamics'][dyn_key][:200]}...")

        count += 1
else:
    print("No backup found - run the restyling cell first.")

## 6. Restore Original (if needed)

Run this cell only if you want to undo all restyling and restore the original.

In [ ]:
# Uncomment to restore the original file:

# backup_file = os.path.join(DRIVE_FOLDER, "tarot_data_full_backup.json")
# if os.path.exists(backup_file):
#     shutil.copy2(backup_file, INPUT_FILE)
#     # Clear restyle checkpoint
#     restyle_cp = os.path.join(DRIVE_FOLDER, "restyle_checkpoint.json")
#     if os.path.exists(restyle_cp):
#         os.remove(restyle_cp)
#     print(f"Restored original from backup!")
#     print(f"Reload the notebook to pick up the restored data.")
# else:
#     print("No backup found.")